# Reading and plotting signals
Read recorded signals from ADI Study Watch and BioNomadix devices. 
Plot ECG, PPG, and accelerometer data from the smartwatch, and ECG from BioNomadix.
Signals must be plotted with x-axis in time, in seconds.

In [140]:
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
import scipy


# Time domain analysis

In [135]:
from datetime import timedelta
# Import Pandas only for reading Watch CSV File and for plotting description in Q1
import pandas as pd


def load_bp_csv(filename, start_datetime):
    test = pd.read_csv(filename, skiprows=8, header=1, delimiter='\t')
    test = test.drop(0)
    test = test.drop(test.columns[3], axis=1)  # Drop the first column
    test = test.rename(columns={'milliSec': 'time_ms', 'CH13': 'ecg', 'CH1': 'ppg'})
    test['ecg'] = pd.to_numeric(test['ecg'], errors='coerce')
    test['timestamp'] = pd.to_datetime(start_datetime) + pd.to_timedelta(test['time_ms'], unit='ms')
    return test


def load_watch_ecg(filename):
    df = pd.read_csv(filename, skiprows=2)
    df = df[['Timestamp', 'ECG data']].dropna()
    df.rename(columns={'Timestamp': 'timestamp', 'ECG data': 'ecg'}, inplace=True)
    df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')
    df['ecg'] = pd.to_numeric(df['ecg'], errors='coerce')
    df = df.dropna()

    # Convert to seconds
    df['time_ms'] = (df['timestamp'] - df['timestamp'].iloc[0])
    return df


def load_watch_data(ecg_file, ppg_file):
    ecg_df = pd.read_csv(ecg_file, skiprows=2)
    ppg_df = pd.read_csv(ppg_file, skiprows=2)

    # Rename columns for clarity
    ecg_df.rename(columns={'Timestamp': 'timestamp', 'ECG data': 'ecg'}, inplace=True)
    ppg_df.rename(columns={'PPG Timestamp': 'timestamp', 'PPG data': 'ppg'}, inplace=True)

    # Add column with time in milli seconds
    ecg_df['time_ms'] = (ecg_df['timestamp'] - ecg_df['timestamp'].iloc[0])
    ppg_df['time_ms'] = (ppg_df['timestamp'] - ppg_df['timestamp'].iloc[0])

    latest_start_time = max(ecg_df['timestamp'].iloc[0], ppg_df['timestamp'].iloc[0])
    earliest_end_time = min(ecg_df['timestamp'].iloc[-1], ppg_df['timestamp'].iloc[-1])

    # Filter data to the common time range
    ecg_df = ecg_df[(ecg_df['timestamp'] >= latest_start_time) & (ecg_df['timestamp'] <= earliest_end_time)]
    ppg_df = ppg_df[(ppg_df['timestamp'] >= latest_start_time) & (ppg_df['timestamp'] <= earliest_end_time)]

    # Drop all columns except timestamp, time_ms and ecg/ppg
    ppg_df = ppg_df.drop(columns=['ADXL Timestamp', 'X', 'Y', 'Z'])
    ecg_df = ecg_df.drop(columns=['Seq No.'])
    # Merge ECG and PPG based on the milli second given
    merged_df = pd.merge_asof(ecg_df.sort_values('timestamp'), ppg_df.sort_values('timestamp'), on='timestamp',
                              direction='nearest')

    merged_df['timestamp'] = pd.to_datetime(merged_df['timestamp'], unit='ms')
    return merged_df


def plot_watch_bp_ecg(watch_df, bp_data):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    # Plot ECG from Watch
    watch_df.sort_values('time_ms_x', inplace=True)
    bp_data.sort_values('time_ms', inplace=True)
    latest_start_time = max(watch_df['timestamp'].iloc[0], bp_data['timestamp'].iloc[0]) + timedelta(minutes=5)
    earliest_end_time = min(watch_df['timestamp'].iloc[-1], bp_data['timestamp'].iloc[-1])

    watch_df = watch_df[(watch_df['timestamp'] >= latest_start_time) & (watch_df['timestamp'] <= earliest_end_time)]
    bp_data = bp_data[(bp_data['timestamp'] >= latest_start_time) & (bp_data['timestamp'] <= earliest_end_time)]

    ax1.plot(watch_df['timestamp'], watch_df['ecg'], label='ECG from Watch', color='blue')
    ax1.set_ylabel('ECG Amplitude')
    ax1.set_title('ECG Signal from ADI Study Watch')
    ax1.legend()
    # Plot ECG from BioNomadix
    ax2.plot(bp_data['timestamp'], bp_data['ecg'], label='ECG from BioNomadix', color='orange')
    ax2.set_xlabel('Time (s)')
    ax2.set_ylabel('ECG Amplitude')
    ax2.set_title('ECG Signal from BioNomadix')
    ax2.legend()
    plt.tight_layout()
    plt.show()



# Time domain
Consider an ECG signal, using fixed windows of 5s with 50% overlap, compute the following features for a segment of 30s long:
- Mean
- Median
- Variance
- Std Dev
- Skewness
- RMS
- Zero-Crossing Rate
- Energy
- Power

Apply this porcedure to the ECG signals acquired from ADI Study Watch and BioNomadix.

In [134]:
bp_clean = load_bp_csv('data/assignment_2/recording_1_ecgppg_biopac_clean_data.txt',
                       start_datetime=datetime(2025, 5, 22, 9, 43, 51, 760000) - timedelta(seconds=0.75))
bp_noisy = load_bp_csv('data/assignment_2/recording_2_ecgppg_biopac_noisy.txt',
                       start_datetime=datetime(2025, 5, 22, 9, 54, 40, 285000) - timedelta(seconds=0.4))
watch_clean = load_watch_data('data/assignment_2/recording_1_ecg_watch_clean_data.csv',
                              'data/assignment_2/recording_1_ppg_watch_clean_data.csv')
watch_noisy = load_watch_data('data/assignment_2/recording_2_ecg_watch_noisy.csv',
                              'data/assignment_2/recording_2_ppg_watch_noisy.csv')

/var/folders/ys/lwj08b754tq4r7w47stwwlx00000gp/T/ipykernel_7656/91148625.py:6: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  test = pd.read_csv(filename, skiprows=8, header=1, delimiter='\t')
/var/folders/ys/lwj08b754tq4r7w47stwwlx00000gp/T/ipykernel_7656/91148625.py:6: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  test = pd.read_csv(filename, skiprows=8, header=1, delimiter='\t')


In [147]:
def plot_data_descriptions(watch_df, bp_df, dataset_name='Give Name Please'):
    import numpy as np
    description_df.loc[len(description_df)] = [
        dataset_name + ' Watch',
        watch_df['ecg'].mean(),
        watch_df['ecg'].median(),
        watch_df['ecg'].var(),
        watch_df['ecg'].std(),
        watch_df['ecg'].skew(),
        np.sqrt(np.mean(watch_df['ecg'] ** 2)),
        np.mean(np.diff(np.sign(watch_df['ecg'])) != 0),
        np.sum(watch_df['ecg'] ** 2),
        np.mean(watch_df['ecg'] ** 2)
    ]
    description_df.loc[len(description_df)] = [
        dataset_name + ' BP',
        bp_df['ecg'].mean(),
        bp_df['ecg'].median(),
        bp_df['ecg'].var(),
        bp_df['ecg'].std(),
        bp_df['ecg'].skew(),
        np.sqrt(np.mean(bp_df['ecg'] ** 2)),
        np.mean(np.diff(np.sign(bp_df['ecg'])) != 0),
        np.sum(bp_df['ecg'] ** 2),
        np.mean(bp_df['ecg'] ** 2)
    ]

In [148]:
# Data Description DF
description_df = pd.DataFrame({
    'Dataset Name': [], 'Mean': [], 'Median': [], 'Variance': [], 'Std Dev': [], 'Skewness': [], 'RMS': [], 'Zero-Crossing Rate': [],
    'Energy': [], 'Power': []
})

plot_data_descriptions(watch_noisy, bp_noisy, dataset_name='Noisy')
plot_data_descriptions(watch_clean, bp_clean, dataset_name='Clean')


In [149]:
description_df

,Dataset Name,Mean,Median,Variance,Std Dev,Skewness,RMS,Zero-Crossing Rate,Energy,Power
0,Noisy Watch,31973.175029,31754.000000,6.272637e+06,2504.523322,5.820849,32071.116893,0.000000,3.267632e+14,1.028557e+09
1,Noisy BP,0.002035,0.108032,3.848639e-01,0.620374,-1.005963,0.620376,0.093165,1.198167e+05,3.848668e-01
2,Clean Watch,32121.565992,31732.000000,8.342445e+06,2888.329027,5.480846,32251.161539,0.000000,3.367923e+14,1.040137e+09
3,Clean BP,0.000445,-0.010986,1.917246e-02,0.138465,9.358622,0.138465,0.022469,6.476600e+03,1.917260e-02


# Frequency domain analysis

In [13]:
# Q2

# Frequency domain 
From the same ECG signals of the previous item, compute the Discrete Fourier Transform and the power spectral density using the functions of periodogram and welch of the scipy.signal library.

In [14]:
# Q3

# Time-frequency domain

In [15]:
# Q4

In [16]:
import numpy as np
import matplotlib.pyplot as plt


def generate_stationary_signals(fs, duration, frequencies):
    t = np.linspace(0, duration, int(fs * duration), endpoint=False)
    return [np.sin(2 * np.pi * freq * t) for freq in frequencies]


def generate_composite_signal(fs, duration, frequencies):
    signal_comp = np.sum(np.stack(generate_stationary_signals(fs, duration, frequencies)), axis=0)
    return signal_comp


def generate_non_stationary_signals(fs,  # Sampling frequency (Hz)
                                    frequencies,  # Frequencies to generate
                                    cycles):
    # Calculate durations and create time array
    segment_durations = [c / f for c, f in zip(cycles, frequencies)]
    total_duration = sum(segment_durations)
    t = np.linspace(0, total_duration, int(fs * total_duration), endpoint=False)
    signal_non_st = np.zeros(len(t))

    # Generate a Non-Stationary signal 
    current_sample = 0
    for freq, n_cycles in zip(frequencies, cycles):
        seg_samples = int(n_cycles / freq * fs)
        local_t = np.arange(seg_samples) / fs
        segment = np.sin(2 * np.pi * freq * local_t)
        signal_non_st[current_sample:current_sample + seg_samples] = segment
        current_sample += seg_samples

    # Create reversed signal
    reversed_signal_non_st = signal_non_st[::-1]

    return signal_non_st, reversed_signal_non_st


stationary_signals = generate_stationary_signals(fs=500, duration=1, frequencies=[10, 25, 50, 100])
comp_signal = generate_composite_signal(fs=500, duration=1, frequencies=[10, 25, 50, 100])
singal_non_st, reversed_signal_non_st = generate_non_stationary_signals(fs=500,  # Sampling frequency (Hz)
                                                                        frequencies=[10, 25, 50, 100],
                                                                        # Frequencies to generate
                                                                        cycles=[3, 6, 12, 25])


In [17]:
# Q5

# Time-frequency-domain analysis

In [18]:
# Q6 

In [19]:
# Q7

In [20]:
# Q8

In [21]:
# Q9

In [22]:
# Q10